In [6]:
import pandas as pd
import html
from typing import List, Dict, Union, Optional
from pathlib import Path

def excel_to_bootstrap_html(
    path: str,
    sheet: Union[str, int, None] = 0,
    *,
    widths: Optional[Union[List[Union[str,float,int]], Dict[str, Union[str,float,int]]]] = None,
    # widths: Liste/Dikt pro Spalte; Einträge dürfen CSS-Strings ("120px","14ch","20%") oder Zahlen sein
    width_unit: str = "px",         # Einheit für numerische widths (z.B. "px", "ch", "em", "%")
    caption: Optional[str] = None,
    table_id: Optional[str] = None,
    embed_bootstrap: bool = True,
    classes: str = "table table-sm table-striped table-hover table-bordered align-middle",
    save_to: Optional[str] = None,
    container_align: str = "left",  # "left" | "center" | "right" (horizontale Position der Tabelle)
    max_table_width: Optional[str] = None,   # z.B. "1200px" oder "80ch" (optional)
) -> str:
    """
    Liest Excel/CSV und rendert eine Bootstrap-HTML-Tabelle.
    - Spaltenbreiten werden NICHT auf 100% normiert. Gib echte CSS-Einheiten ("px","ch","%") oder Zahlen (interpretiert als width_unit) an.
    - Tabelle wird als inline-table mit width:auto gerendert => füllt NICHT automatisch die gesamte Seitenbreite.
    - container_align steuert die horizontale Position (links/zentriert/rechts).
    """
    # 1) Daten laden
    if path.lower().endswith((".csv", ".tsv")):
        sep = "\t" if path.lower().endswith(".tsv") else ","
        df = pd.read_csv(path, sep=sep)
    else:
        df = pd.read_excel(path, sheet_name=sheet)

    if df.shape[1] == 0:
        raise ValueError("Die geladene Tabelle hat keine Spalten.")
    
    col_names = list(map(str, df.columns))
    n_cols = len(col_names)
    tid = table_id or f"tbl_{abs(hash(tuple(col_names)))%10**8}"

    # 2) Spaltenbreiten aufbereiten (KEIN Normalisieren)
    def _as_css(v):
        if isinstance(v, (int, float)):
            return f"{v}{width_unit}"
        v = str(v).strip()
        # wenn keine Einheit enthalten ist, fallback auf width_unit
        if any(v.endswith(u) for u in ("px","%","em","rem","ch","vw","vh")):
            return v
        return f"{v}{width_unit}"

    col_width_css: List[str] = []
    if widths is None:
        col_width_css = []  # kein colgroup -> natürliche Breite / Inhalte entscheiden
    elif isinstance(widths, dict):
        assigned = [widths.get(c, None) for c in col_names]
        col_width_css = [_as_css(w) if w is not None else None for w in assigned]
    else:
        wlist = list(widths)
        # auf n_cols matchen (ohne Normierung)
        if len(wlist) < n_cols:
            wlist = wlist + [None]*(n_cols - len(wlist))
        elif len(wlist) > n_cols:
            wlist = wlist[:n_cols]
        col_width_css = [_as_css(w) if w is not None else None for w in wlist]

    # 3) Head/CSS
    head_parts = []
    if embed_bootstrap:
        head_parts.append(
            '<link rel="stylesheet" '
            'href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css">'
        )

    align_css = {
        "left":   "text-align:left;",
        "center": "text-align:center;",
        "right":  "text-align:right;",
    }.get(container_align.lower(), "text-align:left;")

    table_extra_style = [
        "display:inline-table",   # verhindert 100%-Breite
        "width:auto",             # benutze Inhalt/colgroup
        "table-layout:auto",      # respektiert colgroup mit absoluten Breiten gut
        # Hinweis: bei Prozent-Spaltenbreiten wirken die Prozente relativ zur Tabellenbreite.
    ]
    if max_table_width:
        table_extra_style.append(f"max-width:{max_table_width}")
    table_style_str = ";".join(table_extra_style)

    head_parts.append(f"""
<style>
/* gleiche Zell-Ausrichtung überall sicherstellen */
#{tid} td, #{tid} th {{ text-align:left !important; }}
#{tid} thead th {{ position: sticky; top: 0; z-index: 2; background: var(--bs-body-bg, #fff); }}
#wrap_{tid} {{ {align_css} }}  /* horizontale Positionierung der Tabelle */
</style>
""".strip())

    # 4) Caption
    caption_html = f"<caption class='caption-top'>{html.escape(str(caption))}</caption>" if caption else ""

    # 5) colgroup (nur setzen, wenn Breiten angegeben)
    if col_width_css:
        colgroup = "<colgroup>" + "".join([
            f"<col style='width:{w};'>" if w is not None else "<col>"
            for w in col_width_css
        ]) + "</colgroup>"
    else:
        colgroup = ""  # keine festen Breiten -> natürliche Breite

    # 6) thead/tbody
    thead_html = "<thead><tr>" + "".join([f"<th scope='col'>{html.escape(str(c))}</th>" for c in col_names]) + "</tr></thead>"
    body_rows = []
    for _, row in df.iterrows():
        tds = []
        for c in col_names:
            val = "" if pd.isna(row[c]) else str(row[c])
            tds.append(f"<td>{html.escape(val)}</td>")
        body_rows.append("<tr>" + "".join(tds) + "</tr>")
    tbody_html = "<tbody>" + "".join(body_rows) + "</tbody>"

    # 7) Tabelle (ohne .table-responsive, um 100%-Breite zu vermeiden)
    table_html = (
        f"<div id='wrap_{tid}'>"
        f"<table id='{tid}' class='{classes}' style='{table_style_str}'>"
        f"{caption_html}{colgroup}{thead_html}{tbody_html}"
        f"</table></div>"
    )

    html_out = "\n".join(head_parts + [table_html])

    if save_to:
        p = Path(save_to)
        p.parent.mkdir(parents=True, exist_ok=True)
        with open(p, "w", encoding="utf-8") as f:
            f.write(html_out)

    return html_out

from IPython.display import HTML, display

html_str = excel_to_bootstrap_html(
    r"K:\Team\Böhmer_Michael\PAPER\Ergebnisse\Domain_shift\distribution_compare.xlsx",
    sheet="DomainShift",
    widths=["40ch","15ch","15ch","15ch","15ch","20ch","20ch","20ch", "20ch"],  # echte CSS-Einheiten pro Spalte
    caption="Table 1. External performance",
    container_align="left",        # links positionieren
    max_table_width=None,          # optional: z.B. "1200px"
    save_to=r"K:\Team\Böhmer_Michael\PAPER\Ergebnisse\Domain_shift\distribution_compare.html"
)
display(HTML(html_str))


Feature,DC Shapiro-p,VC Shapiro-p,ΔMean (DC−VC),"KS-p (DC, VC)"
inv cmj uni av. propulsive force,0.779,0.262,-6553.0,1.85e-40
uninv cmj uni av. propulsive force,0.436,0.106,-6615.0,1.85e-40
uninv cmj uni peak braking force,0.76,0.161,-6451.0,5.42e-31
inv cmj uni peak braking force,0.151,0.00897,-6611.0,4.82e-33
uninv cmj uni peak landing force,7.45e-05,0.256,1944.0,9.76e-25
inv cmj uni av. braking force,0.0528,7.72e-10,-1952.0,8.57e-22
inv cmj uni peak landing force,0.00563,0.000169,1428.0,2.32e-17
uninv cmj uni av. braking force,0.436,1.23e-09,-1664.0,5.41e-20
uninv cmj uni av. braking power,0.00764,0.766,2745.0,3.82e-13
inv cmj uni av. braking power,0.0255,0.338,2454.0,2.02e-14


In [17]:
import pandas as pd
import html
from typing import List, Dict, Union, Optional
from pathlib import Path
from IPython.display import HTML, display

def excel_to_bootstrap_html(
    path: str,
    sheet: Union[str, int, None] = 0,
    *,
    widths: Optional[Union[List[Union[str,float,int]], Dict[str, Union[str,float,int]]]] = None,
    width_unit: str = "px",
    caption: Optional[str] = None,
    table_id: Optional[str] = None,
    embed_bootstrap: bool = True,
    classes: str = "table table-sm table-striped table-hover table-bordered align-middle",
    save_to: Optional[str] = None,
    save_standalone: bool = True,      # ← NEU: beim Speichern vollständiges HTML-Dokument erzeugen
    container_align: str = "left",
    max_table_width: Optional[str] = None,
) -> str:
    # ---------------- Daten laden ----------------
    if path.lower().endswith((".csv", ".tsv")):
        sep = "\t" if path.lower().endswith(".tsv") else ","
        df = pd.read_csv(path, sep=sep)
    else:
        df = pd.read_excel(path, sheet_name=sheet)

    if df.shape[1] == 0:
        raise ValueError("Die geladene Tabelle hat keine Spalten.")

    col_names = list(map(str, df.columns))
    n_cols = len(col_names)
    tid = table_id or f"tbl_{abs(hash(tuple(col_names)))%10**8}"

    # --------------- Breiten (keine Normierung) ---------------
    def _as_css(v):
        if v is None:
            return None
        if isinstance(v, (int, float)):
            return f"{v}{width_unit}"
        v = str(v).strip()
        if any(v.endswith(u) for u in ("px","%","em","rem","ch","vw","vh")):
            return v
        return f"{v}{width_unit}"

    if widths is None:
        col_width_css: List[Optional[str]] = [None]*n_cols
    elif isinstance(widths, dict):
        assigned = [widths.get(c, None) for c in col_names]
        col_width_css = [_as_css(w) for w in assigned]
    else:
        wlist = list(widths)
        if len(wlist) < n_cols:
            wlist += [None]*(n_cols-len(wlist))
        elif len(wlist) > n_cols:
            wlist = wlist[:n_cols]
        col_width_css = [_as_css(w) for w in wlist]

    # ----------------- Head/CSS -------------------
    head_parts = []
    if embed_bootstrap:
        head_parts.append(
            '<link rel="stylesheet" '
            'href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css">'
        )

    align_css = {
        "left":   "text-align:left;",
        "center": "text-align:center;",
        "right":  "text-align:right;",
    }.get(container_align.lower(), "text-align:left;")

    table_extra_style = [
        "display:inline-table",
        "width:auto",
        "table-layout:auto",
    ]
    if max_table_width:
        table_extra_style.append(f"max-width:{max_table_width}")
    table_style_str = ";".join(table_extra_style)

    head_parts.append(f"""
<style>
#{tid} td, #{tid} th {{ text-align:left !important; }}
#{tid} thead th {{ position: sticky; top: 0; z-index: 2; background: var(--bs-body-bg, #fff); }}
#wrap_{tid} {{ {align_css} }}
</style>
""".strip())

    head_html = "\n".join(head_parts)

    # ---------------- Table HTML ------------------
    caption_html = f"<caption class='caption-top'>{html.escape(str(caption))}</caption>" if caption else ""
    colgroup = (
        "<colgroup>" + "".join(
            [f"<col style='width:{w};'>" if w else "<col>" for w in col_width_css]
        ) + "</colgroup>"
    ) if col_width_css else ""

    thead_html = "<thead><tr>" + "".join(
        [f"<th scope='col'>{html.escape(str(c))}</th>" for c in col_names]
    ) + "</tr></thead>"

    body_rows = []
    for _, row in df.iterrows():
        tds = [f"<td>{html.escape('' if pd.isna(row[c]) else str(row[c]))}</td>" for c in col_names]
        body_rows.append("<tr>" + "".join(tds) + "</tr>")
    tbody_html = "<tbody>" + "".join(body_rows) + "</tbody>"

    table_only_html = (
        f"<div id='wrap_{tid}'>"
        f"<table id='{tid}' class='{classes}' style='{table_style_str}'>"
        f"{caption_html}{colgroup}{thead_html}{tbody_html}"
        f"</table></div>"
    )

    # Für Notebook-Display: Head + Snippet (wie zuvor)
    html_out = "\n".join([head_html, table_only_html])

    # --------------- Speichern (robuster) ---------------
    if save_to:
        p = Path(save_to)
        p.parent.mkdir(parents=True, exist_ok=True)
        if save_standalone:
            # Vollständiges, eigenständiges HTML-Dokument
            doc = f"""<!doctype html>
<html lang="de">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
{head_html}
<title>Table Export</title>
</head>
<body>
{table_only_html}
</body>
</html>"""
            with open(p, "w", encoding="utf-8") as f:
                f.write(doc)
        else:
            # Nur das bisherige Fragment (wie vorher)
            with open(p, "w", encoding="utf-8") as f:
                f.write(html_out)

    return html_out

html_str = excel_to_bootstrap_html(
    r"K:\Team\Böhmer_Michael\PAPER\Datensätze_final\Datenübersicht\CMJ_ISO_Mean_SD_korrigiert - Kopie.xlsx",
    sheet="Tabelle1",
    widths=["20ch","14ch","10ch","30ch","18ch"],  # echte CSS-Breiten
    caption="Table 1. External performance",
    container_align="left",
    save_to=r"K:\Team\Böhmer_Michael\PAPER\Datensätze_final\Datenübersicht\tabelle.html",  
    save_standalone=True                               
)
display(HTML(html_str))

Features,Motum (Injured),Motum (Uninjured),Maestroni (Injured),Maestroni (Uninjured)
Sample size,n = 32,n = 34,n = 36,n = 35
Demographic features,,,,
Age (y),21.344 ± 4.382,25.794 ± 3.540,23.286 ± 4.144,23.856 ± 2.817
Mass (kg),73.329 ± 13.048,78.111 ± 11.923,71.181 ± 10.208,71.614 ± 6.264
Height (m),1.796 ± 0.052,1.786 ± 0.088,1.723 ± 6.942,1.738 ± 5.360
CMJ Braking features,,,,
INV_CMJ_uni_Peak braking force,9.985 ± 1.784,11.374 ± 1.648,17.113 ± 2.547,17.609 ± 1.889
UNINV_CMJ_uni_Peak braking force,11.299 ± 1.778,10.999 ± 1.847,18.007 ± 2.074,17.254 ± 2.192
INV_CMJ_uni_Av. braking force,7.394 ± 1.471,8.464 ± 1.356,10.050 ± 0.464,9.810 ± 0.070
UNINV_CMJ_uni_Av. braking force,8.405 ± 1.389,8.113 ± 1.539,10.051 ± 0.466,9.825 ± 0.079


### Übersicht Motum und Maestroni

In [14]:
from __future__ import annotations
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.stats import rankdata, kruskal
import scikit_posthocs as sp

# =========================
# === KONFIGURATION =======
# =========================

INPUT_XLSX  = r"K:\Team\Böhmer_Michael\PAPER\Datensätze_final\Statistik\ML_statistik_ganz.xlsx"  # input-datei
SHEET_NAME  = "Tabelle1"                          # oder None für erstes Tabellenblatt
OUTPUT_XLSX = None                                # None => auto: "<input>_dscf.xlsx" neben INPUT_XLSX

# Erwartete Gruppenlabels (exakte Schreibweise!)
EXPECTED_LABELS = [
    "Motum (Injured)",
    "Motum (Uninjured)",
    "Maestroni (Injured)",
    "Maestroni (Uninjured)",
]

# Die 4 geplanten Paarvergleiche (in der gewünschten Reihenfolge)
PAIRS = [
    ("Motum (Injured)", "Motum (Uninjured)"),
    ("Motum (Injured)", "Maestroni (Injured)"),
    ("Motum (Uninjured)", "Maestroni (Uninjured)"),
    ("Maestroni (Injured)", "Maestroni (Uninjured)"),
]

# hübsches p-Format
def pformat(p: float) -> str:
    if pd.isna(p):
        return "NA"
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def main():
    
    # === Pfade prüfen ===
    in_path = Path(INPUT_XLSX).expanduser()
    if not in_path.exists():
        raise FileNotFoundError(f"Eingabedatei nicht gefunden: {in_path}")

    if OUTPUT_XLSX is None or str(OUTPUT_XLSX).strip() == "":
        out_path = in_path.with_name(in_path.stem + "_dscf.xlsx")
    else:
        out_path = Path(OUTPUT_XLSX).expanduser()
        if out_path.suffix.lower() != ".xlsx":
            out_path = out_path.with_suffix(".xlsx")

    print(f"[INFO] Lese:   {in_path.resolve()}")
    print(f"[INFO] Sheet:  {SHEET_NAME if SHEET_NAME else '(erstes Tabellenblatt)'}")
    print(f"[INFO] Schreibe:{out_path.resolve()}")

    # === Excel einlesen ===
    try:
        df = pd.read_excel(in_path, sheet_name=SHEET_NAME)
    except Exception:
        print("[FEHLER] Konnte Excel nicht lesen.")
        raise

    if df.shape[1] < 5:
        raise ValueError("Erwartet: mind. 5 Spalten (1 Gruppe + >=4 Features).")

    # Gruppenspalte + Feature-Spalten
    group_col = df.columns[0]
    groups = df[group_col].astype(str)
    feature_cols = df.columns[1:]

    # Labels prüfen (Warnung, kein harter Fehler)
    observed = set(groups.unique().tolist())
    missing = [lab for lab in EXPECTED_LABELS if lab not in observed]
    if missing:
        print("[WARNUNG] Einige erwartete Labels fehlen (exakte Schreibweise prüfen):")
        for m in missing:
            print(f"  - {m}")
        print("Beobachtete Labels:", observed)

    # Header für die Ausgabe (DSCF statt Holm)
    header = [
        "Feature", "Kruskal Wallis H", "Kruskal Wallis p",
        "Motum (Injured) vs Motum (Uninjured) p (DSCF)",
        "Motum (Injured) vs Maestroni (Injured) p (DSCF)",
        "Motum (Uninjured) vs Maestroni (Uninjured) p (DSCF)",
        "Maestroni (Injured) vs Maestroni (Uninjured) p (DSCF)",
    ]

    results_rows = []
    debug_rows = []
    g_all = groups.to_numpy()

    # === pro Feature: Kruskal–Wallis + DSCF-Posthoc (4 Paare) ===
    for feat in feature_cols:
        # 1) sauber numerisieren & Maske
        vals = pd.to_numeric(df[feat], errors="coerce")
        mask_valid = vals.notna() & groups.isin(EXPECTED_LABELS)
        x_clean = vals[mask_valid].to_numpy()
        g_clean = g_all[mask_valid]

        # 2) Omnibus (KW) – gefiltert
        groups_list = [
            vals[(groups == lab) & vals.notna()].to_numpy()
            for lab in EXPECTED_LABELS
            if ((groups == lab) & vals.notna()).any()
        ]
        if sum(len(arr) > 0 for arr in groups_list) >= 2:
            H_kw, p_kw = kruskal(*groups_list)
        else:
            H_kw, p_kw = np.nan, np.nan

        # 3) DSCF-Posthoc über alle Gruppen (familienweise FWER-Kontrolle)
        #    scikit-posthocs akzeptiert tidy-DF: Spalten "val" & "grp"
        tidy = pd.DataFrame({"val": x_clean, "grp": g_clean})
        # p_adjust=None: DSCF selbst kontrolliert; keine zusätzliche Korrektur!
        dscf_mat = sp.posthoc_dscf(tidy, val_col="val", group_col="grp")

        # 4) p-Werte für die 4 gewünschten Paare extrahieren
        pair_ps = []
        for (a, b) in PAIRS:
            try:
                p = float(dscf_mat.loc[a, b])
            except KeyError:
                # Falls eine der Gruppen im aktuellen Feature fehlt
                p = np.nan
            pair_ps.append(p)

        # ---- DEBUG-Infos sammeln (N je Gruppe, Mittelränge, DSCF-roh p) ----
        ranks = rankdata(x_clean) if x_clean.size else np.array([])
        mean_ranks = {lab: (float(ranks[g_clean == lab].mean()) if (g_clean == lab).any() else np.nan)
                      for lab in EXPECTED_LABELS}
        sizes = {lab: int(np.sum(g_clean == lab)) for lab in EXPECTED_LABELS}

        debug_entry = {
            "Feature": str(feat),
            "KW_H": float(H_kw) if np.isfinite(H_kw) else np.nan,
            "KW_p": float(p_kw) if np.isfinite(p_kw) else np.nan,
        }
        for lab in EXPECTED_LABELS:
            debug_entry[f"N_{lab}"] = sizes.get(lab, np.nan)
            debug_entry[f"MeanRank_{lab}"] = mean_ranks.get(lab, np.nan)
        for (a, b), p in zip(PAIRS, pair_ps):
            debug_entry[f"DSCF_p: {a} vs {b}"] = p if np.isfinite(p) else np.nan
        debug_rows.append(debug_entry)
        # ---- DEBUG Ende ----

        # 5) Ergebniszeile
        row = [
            str(feat),
            f"{H_kw:.3f}" if np.isfinite(H_kw) else "NA",
            pformat(p_kw),
        ] + [pformat(p) for p in pair_ps]
        results_rows.append(row)

    out_df = pd.DataFrame(results_rows, columns=header)
    debug_df = pd.DataFrame(debug_rows)

    # === Excel schreiben: zwei Sheets (Results + DEBUG) ===
    with pd.ExcelWriter(out_path, engine="openpyxl") as xlw:
        out_df.to_excel(xlw, sheet_name="Results", index=False)
        debug_df.to_excel(xlw, sheet_name="DEBUG", index=False)

    print(f"[OK] Fertig. Datei geschrieben: {out_path.resolve()}")

if __name__ == "__main__":
    try:
        main()
    except ImportError as e:
        print("[FEHLER] Fehlendes Paket. Bitte installieren:")
        print("    pip install pandas scipy openpyxl scikit-posthocs")
        sys.exit(1)


[INFO] Lese:   \\server01-fs\data\Team\Böhmer_Michael\PAPER\Datensätze_final\Statistik\ML_statistik_ganz.xlsx
[INFO] Sheet:  Tabelle1
[INFO] Schreibe:\\server01-fs\data\Team\Böhmer_Michael\PAPER\Datensätze_final\Statistik\ML_statistik_ganz_dscf.xlsx
[OK] Fertig. Datei geschrieben: \\server01-fs\data\Team\Böhmer_Michael\PAPER\Datensätze_final\Statistik\ML_statistik_ganz_dscf.xlsx


### Statistik Gruppenvergleich (ANOVA, Welch ANOVA, Kruskal Wallis)

In [4]:
from __future__ import annotations
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.stats import (
    rankdata, kruskal, f_oneway, shapiro, levene, tukey_hsd
)
import scikit_posthocs as sp


# =========================
# === KONFIGURATION =======
# =========================

INPUT_XLSX  = r"K:\Team\Böhmer_Michael\PAPER\Datensätze_final\Statistik\ML_statistik_ganz.xlsx"
SHEET_NAME  = "Tabelle1"
OUTPUT_XLSX = r"K:\Team\Böhmer_Michael\PAPER\Datensätze_final\Statistik\ML_statistik_anova.xlsx"

EXPECTED_LABELS = [
    "Motum (Injured)",
    "Motum (Uninjured)",
    "Maestroni (Injured)",
    "Maestroni (Uninjured)",
]

PAIRS = [
    ("Motum (Injured)", "Motum (Uninjured)"),
    ("Motum (Injured)", "Maestroni (Injured)"),
    ("Motum (Uninjured)", "Maestroni (Uninjured)"),
    ("Maestroni (Injured)", "Maestroni (Uninjured)"),
]

def pformat(p: float) -> str:
    if pd.isna(p):
        return "NA"
    return "<0.001" if p < 0.001 else f"{p:.3f}"


# =====================================================================
# ===============   ANOVA & WELCH HELFER   ============================
# =====================================================================

def run_anova(groups_dict):
    arrs = [v for v in groups_dict.values()]
    F, p = f_oneway(*arrs)
    return F, p

def run_welch(groups_dict):
    """
    Berechnet die Welch-ANOVA (F und p-Wert) manuell.
    Quelle: Welch (1951), Brown-Forsythe-Korrektur.
    """
    groups = list(groups_dict.values())
    k = len(groups)

    ns = np.array([len(g) for g in groups], dtype=float)
    means = np.array([np.mean(g) for g in groups], dtype=float)
    vars_ = np.array([np.var(g, ddof=1) for g in groups], dtype=float)

    ws = ns / vars_

    weighted_mean = np.sum(ws * means) / np.sum(ws)

    num = np.sum(ws * (means - weighted_mean) ** 2) / (k - 1)

    denom = 1 + (2 * (k - 2) /
                 (k**2 - 1)) * np.sum((1 / (ns - 1)) *
                 (1 - ws / np.sum(ws)) ** 2)

    F_welch = num / denom

    df1 = k - 1
    df2 = ( (3 / (k**2 - 1)) *
            np.sum((1 / (ns - 1)) * (1 - ws / np.sum(ws)) ** 2)
          ) ** -1

    from scipy.stats import f
    p = 1 - f.cdf(F_welch, df1, df2)

    return F_welch, p



# =====================================================================
# =======================   MAIN   ====================================
# =====================================================================

def main():

    in_path = Path(INPUT_XLSX).expanduser()
    if not in_path.exists():
        raise FileNotFoundError(in_path)

    if OUTPUT_XLSX is None:
        out_path = in_path.with_name(in_path.stem + "_dscf.xlsx")
    else:
        out_path = Path(OUTPUT_XLSX).expanduser()
        if out_path.suffix.lower() != ".xlsx":
            out_path = out_path.with_suffix(".xlsx")

    print(f"[INFO] Output: {out_path}")

    df = pd.read_excel(in_path, sheet_name=SHEET_NAME)
    group_col = df.columns[0]
    groups = df[group_col].astype(str)
    feature_cols = df.columns[1:]
    g_all = groups.to_numpy()

    # Ergebnis-Header
    header = [
        "Feature", "Omnibus_Stat", "Omnibus_p",
        "Motum (Injured) vs Motum (Uninjured) p",
        "Motum (Injured) vs Maestroni (Injured) p",
        "Motum (Uninjured) vs Maestroni (Uninjured) p",
        "Maestroni (Injured) vs Maestroni (Uninjured) p",
    ]

    results_rows = []
    debug_rows = []

    # =====================================================================
    # ======================= PRO FEATURE =================================
    # =====================================================================

    for feat in feature_cols:

        vals = pd.to_numeric(df[feat], errors="coerce")

        # gruppieren
        groups_dict = {
            lab: vals[(groups == lab) & vals.notna()].to_numpy()
            for lab in EXPECTED_LABELS
            if ((groups == lab) & vals.notna()).any()
        }

        if len(groups_dict) < 2:
            results_rows.append([feat, "NA", "NA"] + ["NA"]*4)
            continue

        # ======================================================
        # 1) Shapiro pro Gruppe
        # ======================================================

        shapiro_ps = {}
        shapiro_ok = True

        for lab, arr in groups_dict.items():
            if len(arr) < 3:
                shapiro_ps[lab] = np.nan
                shapiro_ok = False
                continue
            stat, p_sh = shapiro(arr)
            shapiro_ps[lab] = p_sh
            if p_sh < 0.05:
                shapiro_ok = False

        # ======================================================
        # 2) Levene (nur wenn Normalverteilung ok)
        # ======================================================

        levene_p = np.nan
        levene_ok = False

        if shapiro_ok:
            if len(groups_dict) >= 2:
                lev_stat, lev_p = levene(*groups_dict.values())
                levene_p = lev_p
                levene_ok = lev_p >= 0.05

        # ======================================================
        # 3) Entscheidungslogik
        # ======================================================

        if shapiro_ok and levene_ok:
            test_used = "ANOVA"
            Omnibus_stat, Omnibus_p = run_anova(groups_dict)

            if Omnibus_p >= 0.05:
                pair_results = ["a"] * len(PAIRS)
            else:
                tidy = pd.DataFrame({
                    "val": np.concatenate(list(groups_dict.values())),
                    "grp": np.concatenate([[k]*len(v) for k,v in groups_dict.items()])
                })
                tuk = sp.posthoc_tukey(tidy, val_col="val", group_col="grp")

                pair_results = []
                for (a, b) in PAIRS:
                    try: 
                        p = float(tuk.loc[a, b])
                    except:
                        p = np.nan
                    pair_results.append(p)

        elif shapiro_ok and not levene_ok:
            test_used = "WELCH"
            Omnibus_stat, Omnibus_p = run_welch(groups_dict)

            if Omnibus_p >= 0.05:
                # keine signifikanten Gesamtunterschiede -> keine Post-hoc-Tests
                pair_results = ["a"] * len(PAIRS)
            else:
                # Daten in "tidy" Form bringen
                tidy = pd.DataFrame({
                    "val": np.concatenate(list(groups_dict.values())),
                    "grp": np.concatenate([[k] * len(v) for k, v in groups_dict.items()])
                })

                # Post-hoc: Tamhane T2 (geeignet für Normalverteilung + ungleiche Varianzen)
                gh = sp.posthoc_tamhane(tidy, val_col="val", group_col="grp")

                pair_results = []
                for (a, b) in PAIRS:
                    try:
                        p = float(gh.loc[a, b])
                    except Exception:
                        p = np.nan
                    pair_results.append(p)

        # elif shapiro_ok and not levene_ok:
        #     test_used = "WELCH"
        #     Omnibus_stat, Omnibus_p = run_welch(groups_dict)

        #     if Omnibus_p >= 0.05:
        #         pair_results = ["a"] * len(PAIRS)
        #     else:
        #         tidy = pd.DataFrame({
        #             "val": np.concatenate(list(groups_dict.values())),
        #             "grp": np.concatenate([[k]*len(v) for k,v in groups_dict.items()])
        #         })
        #         gh = sp.posthoc_gameshowell(tidy, val_col="val", group_col="grp")

        #         pair_results = []
        #         for (a, b) in PAIRS:
        #             try: 
        #                 p = float(gh.loc[a, b])
        #             except:
        #                 p = np.nan
        #             pair_results.append(p)


        else:
            test_used = "KRUSKAL"
            Omnibus_stat, Omnibus_p = kruskal(*groups_dict.values())

            if Omnibus_p >= 0.05:
                pair_results = ["a"] * len(PAIRS)
            else:
                tidy = pd.DataFrame({
                    "val": np.concatenate(list(groups_dict.values())),
                    "grp": np.concatenate([[k]*len(v) for k,v in groups_dict.items()])
                })
                dscf = sp.posthoc_dscf(tidy, val_col="val", group_col="grp")

                pair_results = []
                for (a, b) in PAIRS:
                    try:
                        p = float(dscf.loc[a, b])
                    except:
                        p = np.nan
                    pair_results.append(p)

        # ======================================================
        # Ergebniszeile (Results-Sheet)
        # ======================================================

        result_row = [
            feat,
            f"{Omnibus_stat:.3f}" if np.isfinite(Omnibus_stat) else "NA",
            pformat(Omnibus_p),
        ] + [r if r == "a" else pformat(r) for r in pair_results]

        results_rows.append(result_row)

        # ======================================================
        # DEBUG-Zeile
        # ======================================================

        debug_entry = {
            "Feature": feat,
            "TestUsed": test_used,
            "Omnibus_Stat": Omnibus_stat,
            "Omnibus_p": Omnibus_p,
            "Normality_OK": shapiro_ok,
            "Levene_p": levene_p,
            "Homogeneity_OK": levene_ok,
        }

        # Shapiro p-Werte hinzufügen
        for lab in EXPECTED_LABELS:
            debug_entry[f"Shapiro_{lab}"] = shapiro_ps.get(lab, np.nan)

        # Paarweise Roh-p’s (zur Validierung)
        for (a, b), pval in zip(PAIRS, pair_results):
            key = f"PosthocRaw: {a} vs {b}"
            debug_entry[key] = pval if pval != "a" else "a"

        debug_rows.append(debug_entry)

    # ======================================================
    #   Excel schreiben
    # ======================================================

    out_df = pd.DataFrame(results_rows, columns=header)
    debug_df = pd.DataFrame(debug_rows)

    with pd.ExcelWriter(out_path, engine="openpyxl") as xlw:
        out_df.to_excel(xlw, sheet_name="Results", index=False)
        debug_df.to_excel(xlw, sheet_name="DEBUG", index=False)

    print(f"[OK] Fertig. Datei geschrieben: {out_path.resolve()}")


if __name__ == "__main__":
    try:
        main()
    except ImportError:
        print("Bitte installieren: pandas scipy openpyxl scikit-posthocs")
        sys.exit(1)


[INFO] Output: K:\Team\Böhmer_Michael\PAPER\Datensätze_final\Statistik\ML_statistik_anova.xlsx
[OK] Fertig. Datei geschrieben: \\server01-fs\data\Team\Böhmer_Michael\PAPER\Datensätze_final\Statistik\ML_statistik_anova.xlsx
